# Silver Exploration Notebook


## Imports

In [ ]:
import os
import findspark

from pyspark.sql import SparkSession

from pyspark.sql.functions import (
    col,
    lit,
    when,
    to_timestamp,
    from_utc_timestamp,
    dayofweek,
    hour,
    month,
    input_file_name,
    regexp_extract,
    arrays_zip,
    explode,
    size,
    count,
    length,
    min,
    max,
    avg,
    slice,
    regexp_extract,
    desc,
    unix_micros,
    abs,
    split,
    element_at,
    countDistinct,
    percentile_approx,
    radians,
    sin,
    cos,
    asin,
    sqrt,
    pow,
    when,
    sum as spark_sum
)

## Start Spark

In [ ]:
findspark.init()

spark = (
    SparkSession.builder
    .appName("silver-exploration")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)

In [ ]:
RAW = os.path.expanduser("~/spark-lab-data/dev/project/raw")
print(RAW)

## Read Bike-sharing dataset
+ Adding city and month

In [ ]:
# Bike-sharing data
trips_raw = (
    spark.read
    .option("header", True)
    .csv(f"{RAW}/trips/*/*.csv")
    .withColumn("src", input_file_name())
)

trips_raw = (
    trips_raw
    .withColumn("city", element_at(split(col("src"), "/"), -2))
    .withColumn("year_month", element_at(split(col("src"), "/"), -1))
    .withColumn("year_month",split(col("year_month"), r"\.").getItem(0))
)

trips_raw.select("city", "year_month").show(5)

## 1. Exporation of Bike-sharing dataset

In [ ]:
trips_raw.printSchema()
trips_raw.show(5, truncate=False)
print("Total rows:", trips_raw.count())
print("Columns:", trips_raw.columns)

- We have more than 8 470 000 biketrips in Oslo, Trondheim and Bergen.
- The columns in the Bike-sharing dataset include: `started_at`, `endend_at`, `duration`, `start_station_id`, `start_station_name`, `start_station_description`, `start_station_latitude`, `start_station_longitude`, `end_station_id`, `end_station_name`, `end_station_description`, `end_station_latitude`, `end_station_longitude`. We also have the added `src`, `city` and `year_month`.

#### Data Types
The 16 columns are stored as string data types. This is because PySpark reads CSV columns as strings by default. However, Several columns represent numeric values or timestamps. We will therefore convert these columns to appropriate data types. 

- Timestamps: `started_at`and `ended_at` -> timestamp
- Trip duration: `duration` -> integer(maybe double?)
- Coorinates: `latitude`and `longitude`-> double

Maybe:
- StationIDs: `start_station_id` and `stop_station_id` -> keep as string or convert to integer

From the table above we can see the ID is stored as a numeric value, but we would argue to keep them as a string as they are identifiers rather than numeric measurements. This also provides a greater flexibility if future datasets contains letters(e.g. ST123), without requiring changes to the data type. 


In [ ]:
trips_silver = trips_raw

## 2. Converting data types

### duration: string -> double

In [ ]:
trips_silver = (
    trips_silver
    .withColumn("duration", col("duration").cast("double"))
)

# Did the convertion work?
trips_silver.filter(
    col("duration").isNull() 
).count()

### started_at / ended_at: string -> timestamp
The `started_at` and `ended_at` timestamps are recorded in UTC, as indicated by `+00:00`.

For example:
- `started_at`: 2022-08-01 05:14:13.781000+00:00
- `ended_at`: 2022-08-01 05:26:27.585000+00:00

In [ ]:
# Setting the spark session timezone to UTC
spark.conf.set("spark.sql.session.timeZone", "UTC")

# Converting the started_at and ended_at columns to timestamp type
trips_silver = (
    trips_silver
    .withColumn("started_ts", to_timestamp(col("started_at")))
    .withColumn("ended_ts", to_timestamp(col("ended_at")))
)

# Let's compare the converted timestamps
trips_silver.select(
    "started_at",
    "started_ts",
    "ended_at",
    "ended_ts"
).show(10, truncate=False)

# Did the convertion work?
trips_silver.filter(
    col("started_ts").isNull() |
    col("ended_ts").isNull()
).count()

### latitude/ longitude: string -> double

In [ ]:
trips_silver = (
    trips_silver
    .withColumn("start_station_latitude", col("start_station_latitude").cast("double"))
    .withColumn("start_station_longitude", col("start_station_longitude").cast("double"))
    .withColumn("end_station_latitude", col("end_station_latitude").cast("double"))
    .withColumn("end_station_longitude", col("end_station_longitude").cast("double"))
)

# Did the convertion work?
trips_silver.filter(
    col("start_station_latitude").isNull() |
    col("start_station_longitude").isNull() |
    col("end_station_latitude").isNull() |
    col("end_station_longitude").isNull()
).count()


## 3. Missing values and Duplicates

In [ ]:
trips_silver.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in trips_silver.columns
]).show()

#### Missing values
The `start_station_description` column contains 11 000 missing values, while `end_station_description` contains around 7000 missing values.

#### Is this detrimental to our project or analysis?
These columns are not truly important for the type of analysing we want to do. Our main objective is to investigate how weather conditions influence bike-sharing usage. Since station ID and names are available, the missing descriptions do not prevent us from identifying stations and analyzing trips. 

Although they are not required for our current objectives, they could be valuable for other purposes later. In the Gold layer, we can simply exclude columns that are not relevant to are final analytical dataset. 

### Investigating duplicates

In [ ]:
print("Total rows:", trips_silver.count())
print("Duplicate rows:", trips_silver.count() - trips_silver.distinct().count())

Out of the 8 470 330 trips, 25 additional duplicates. 

#### Are we sure these are duplicates?

We can not be certain that these are dublicates because the trips do not contain any identifier. These records do however contain identical entries across all 16 collumns, including timestamp with microseconds precision. It is highly unlikely that separate trips would share all these values.

We consider therefore these records to be duplicate entries, potentially from the source data or overlapping records during the ingestion process. The exact cause has not been investigated further, but we want to remove the additional copies. 

Given that our dataset contain over 8 million trips, removing 25 duplicate records will have a negligible impact on the overall analysis. If the dataset was significanlty smaller, further investigation would be important as removing even a small number of records could have a greater influence on the result.

In [ ]:
duplicate_cols = [
    c for c in trips_silver.columns
    if c not in ["src", "year_month"]
]

trips_silver = trips_silver.dropDuplicates(duplicate_cols)

print("Total rows:", trips_raw.count())
print("Duplicate rows:", trips_raw.count() - trips_silver.count())

## 4. Timestamp Validation

### Investigating the timestamp format

In [ ]:
trips_silver.groupBy(length("started_at").alias("len")).count().orderBy("len").show()
trips_silver.select("started_at", "ended_at", "duration").distinct().limit(10).show(truncate=False)

### Converting Timestamps to Norwegian Local Time

We convert these timestamps to Norwegian local time (`Europe/Oslo`), accounting for daylight saving time. The original UTC timestamps are preserved for further processing.

In [ ]:
trips_silver = (
    trips_silver
    .withColumn(
        "started_local",
        from_utc_timestamp("started_ts", "Europe/Oslo")
    )
    .withColumn(
        "ended_local",
        from_utc_timestamp("ended_ts", "Europe/Oslo")
    )
    
)

trips_silver.select("started_ts","started_local","ended_ts","ended_local").show(10, truncate=False)

#### Does duration correspond to the start and end time?

In [ ]:
trips_silver.select(
    "started_local",
    "ended_local",
    "duration",
    # does started_at and ended_at match the duration?
    (
        (
            unix_micros("ended_ts") -
            unix_micros("started_ts")
        ) / 1000000
    ).alias("duration_check")

).show(5, truncate=False)

From the table above it looks like the duration always round down.

for example:
- 363 and 363.712
- 703 and 703
- 778 and 778.756

Let's investigate the duration a bit further.

In [ ]:
trips_check = trips_silver.withColumn(
    "duration_diff",
    abs(
        (unix_micros("ended_ts")
         - unix_micros("started_ts")) / 1_000_000
        - col("duration").cast("double")
    )
)

trips_check.select("duration_diff").summary().show()

Since the maximum difference are below one second, we consider the recorded durations consistent with the timestamps. 

## 5. Coordinate Validation

#### Valid cordination for latitude and longitude in Norway:
- Norwegian latitude: 57 - 71
- Norwegian longitude: 4 - 31

In [ ]:
trips_silver.filter(
    (col("start_station_latitude") < 57) |
    (col("start_station_latitude") > 71) |
    (col("end_station_latitude") < 57) |
    (col("end_station_latitude") > 71) |
    (col("start_station_longitude") < 4) |
    (col("start_station_longitude") > 31) |
    (col("end_station_longitude") < 4) |
    (col("end_station_longitude") > 31)
).count()

In [ ]:
trips_silver.select(
    min("start_station_latitude").alias("min_lat"),
    max("start_station_latitude").alias("max_lat"),
    min("start_station_longitude").alias("min_lon"),
    max("start_station_longitude").alias("max_lon")
).show()

The latitude and logitude values appear to reasonable based on the geographical locations of the three cities. Oslo is the city most southern, while Trondheim is the city most nothern. Bergen is the city most western, while oslo is the city most eastern. 

|   City       |   latitude    |   longitude   |
|--------------|---------------|---------------|
|  Oslo:       |     59.91     |     10.75     |  
|  Bergen:     |     60.39     |      5.32     |
|  Trondheim:  |     63.43     |     10.40     |

### Number of trips divided by the different cities

In [ ]:
trips_silver.groupBy("city").count().show()

Oslo accounts for approximately 68% of all trips in the dataset, so the analysis of the trips may be dominated by the travel patterns in Oslo.

### Number of trips devided by city and stations

In [ ]:
trips_silver.groupBy("city", "start_station_id", "end_station_id").count().show(200)

From the table above it looks like the stationIDs might use the same identifications accross the cities. So the ID alone won't be possible to identify which city the trip was in.  

Let's investigate this a bit ferther:

In [ ]:
trips_silver.groupBy("start_station_id").agg(
    countDistinct("city").alias("num_cities")
).filter(
    col("num_cities") > 1
).show()

The investigantion shows that no start station id is shared across cities in this current dataset. I do think including cities when grouping by station IDs to ensure robustness against potential ovelaps in future data and it gives a better overview. 

### Number of trips divided by city and month

In [ ]:
trips_silver.groupBy("city", "year_month").count().orderBy(
    "city", "year_month"
).show(200)

## 6. Trip Duration Analysis

In [ ]:
trips_silver.select("duration").summary("min", "25%", "50%", "75%", "max").show()

trips_silver.select(spark_sum(when(col("duration") <= 0, 1).otherwise(0)).alias("non_positive"),
         spark_sum(when(col("duration") < 60, 1).otherwise(0)).alias("under_60s"),
         spark_sum(when(col("duration").isNull(), 1).otherwise(0)).alias("null_duration")).show()

#### Durations to investigate ferther
- 2 trips has a negative or a duration of 0.
- Is it possible that a trip last around 22 hours? (79790/3600 = 22)

In [ ]:
# Trips with durations smaller than 1 second
trips_silver.filter(
    col("duration").cast("double") <= 1
).show(truncate=False)

print("How many trips have a duration of 1 second or less?", trips_silver.filter(col("duration").cast("double") <= 1).count())

### Extremely Short Trips
29 trips with a recorded duration of one second or less, including two trips with a duration of zero seconds. 

With the table above these trips have a different start and end station which makes it physically impossible to complete the journeys within that timeframe.

These records likely represent data quality issues that need futher investigation.

Possible solution: 
- Find the mean/median trip duration from one statiotn to the other and compare it to minimum duration record.
- Flag trips under a spesific timeframe. 
- Anything else?

In [ ]:
station_durations = (
    trips_silver
    .groupBy("city", "start_station_id", "end_station_id")
    .agg(
        count("*").alias("num_trips"),
        avg("duration").alias("mean_duration"),
        min("duration").alias("min_duration"),
        percentile_approx("duration", 0.5).alias("median_duration")
    )
)

station_durations.filter(
    col("num_trips") >= 10
).orderBy("min_duration").show(20, truncate=False)

### What does the table above tell us?
Although the mean and median duration do not accually tell us the minimum possible duration between two stations, it provides a useful indication of what a typical trip looks like. 

The table shows station pairs with hundres and thousands of bike-trips, where the average duration is several minutes, yet the shortest recorded time is only zero or one second. This strongly suggest that some values are incorrect or unrealistic. 

We also see that the mean duration is constantly higher than the median for the station pairs, suggesting than long trips may influance the mean. The median may therefore provide a more robust reference identifying unusually short trips.

On another note this does make me question the dataset and its accuracy. Are these simply a small number of isolated data quality issues, or could they indicate more problems with the data?

For now let's atleast flag trips with a duration smaller than one second. Further investigation is needed to determine how many trips have a duration significantly below the median of their station pairs and whether these records could affect downstream analyses. 

In [ ]:
# Flag trips that are under 1 second
trips_silver = trips_silver.withColumn(
    "short_trip_flag",
    col("duration") <= 1
)

In [ ]:
total_trips = trips_silver.count()

trips_with_median = trips_silver.join(
    station_durations.select(
        "city",
        "start_station_id",
        "end_station_id",
        "median_duration",
        "num_trips"
    ),
    on = ["city", "start_station_id", "end_station_id"],
    how = "left"
)

for threshold in [0.01, 0.05, 0.10, 0.20]:
    short_trips = trips_with_median.filter(
        (col("num_trips") >= 10) &
        (col("median_duration") > 0) &
        (col("duration") < col("median_duration") * threshold)
    ).count()

    percentage = (short_trips / total_trips) * 100

    print(f"Below {threshold:.0%} of median: " 
          f"{short_trips} trips ({percentage:.6f})")

The result: 
- Below 1% of median: 56 trips (0.000661)
- Below 5% of median: 1230 trips (0.014519)
- Below 10% of median: 10995 trips (0.129785)
- Below 20% of median: 36769 trips (0.434021)

This investigatet only station pairs with a minimum of ten recorded trips and a positive median duration.

Since a short trip duration relative to the median does not necessarlity indicate an invalid trip, introducing a higher threshold could incorrectly flag valid trips. 

To invsetigate this further, we can find the straight-line distance between the stations pairs and use the recorded duration to estimate the minimum required average speed. 

In [ ]:
# Converting coordinates from degrees to radians
lat1 = radians(col("start_station_latitude"))
lat2 = radians(col("end_station_latitude"))

lon1 = radians(col("start_station_longitude"))
lon2 = radians(col("end_station_longitude"))

# Haversine formula
a = (
    pow(sin((lat2 - lat1) / 2), 2) +
    cos(lat1) * cos(lat2) *
    pow(sin((lon2 - lon1) / 2), 2)
)

# Calulate straight-lin distance in kilometers
trips_with_distance = trips_with_median.withColumn(
    "distance_km",
     2 * 6371 * asin(sqrt(a))
)


trips_with_distance.select(
    "city",
    "start_station_id",
    "end_station_id",
    "duration",
    "median_duration",
    "distance_km"
).show(20, truncate=False)

In [ ]:
trips_with_speed = trips_with_distance.withColumn(
    "speed_kmh",
    when(
        (col("duration") > 0) &
        (col("distance_km").isNotNull()),
        (col("distance_km") * 3600) / col("duration")
    )
)

trips_with_speed.select(
    "city",
    "start_station_id",
    "end_station_id",
    "duration",
    "median_duration",
    "distance_km",
    "speed_kmh"
).orderBy(
    col("speed_kmh").desc()
).show(20, truncate=False)

#### Unrealistic Cycling speeds
According to vegvesenet, electric bicycles must stop providing motor assistance when reaching a speed of 25 km/h. You may exceed this speed through padelling or downhill cycling. A speed over 100 km/h is highly unlikely, but a speed of around 12 000 km/h must be some time of record. 

Since our calculated speed is based on the straight-line distance between stations, it therefor represent a lower bound for the actual cycling speed. To invetigate potential unrealistic trips, we can evaluate different spees thresholds.

In [ ]:
for threshold in [25, 30, 40, 50, 60, 100]:
    fast_trips = trips_with_speed.filter(
        col("speed_kmh") > threshold
    ).count()

    percentage_speed = (fast_trips / total_trips) * 100

    print(f"Above {threshold} km/h "
          f"{fast_trips} trips ({percentage_speed:.6f}%)")

The result show that 1786 trips har a mimimum average speed exceeding 25 km/h. More notably 874 trips exceed 69 km/h and 567 trips exceeds 100 km/h.

Even though these record represent a small proportion of the dataset, the extreme high estimated speeds strongly suggest data quality issues. 

Compared to the media-based approach, the stright-line distance method provides a more certanty that the trips are in fact unrealistic. 

We could therefore introduce an additional speed-based quality flag to improve the reliability. 

In [ ]:
additional_invalid_trips = trips_silver.filter(
    (col("speed_kmh") > 50) &
    (col("duration") <= 1)
).count()

print(f"Trips that are both a speed over 60 km/h and 1 secound: " f"{additional_invalid_trips}")

We know that there is 29 trips with a duration with one second or less. We also know that there is 1005 trips with a speed over 50 km/h and now we know that 27 of those trips around 1 second are included in the trips that are over a speed of 50 km/h. 

I would therefore conclude with that we can have both instances, both flag trips around one second and flag trips with a speed over 50 km/h. 

In [ ]:
# Flag trips that have a speed over 50 km/h
trips_silver = trips_silver.withColumn(
    "unrealistic_speed_flag",
    col("speed_kmh") > 50
)

### Trips with a long duration

In [ ]:
print("Number of trips longer than 12 hours: ", trips_silver.filter(col("duration") > 12 * 3600).count())

trips_silver.select(
    "started_at",
    "ended_at",
    "duration"
).orderBy(desc("duration")).show(15, truncate=False)

Could a city bike realistically be used for appocimately 22 hours?

Accoring to Oslo Bysykkel, a fully charged electric bike has an estimate range of around 70 kilometers. The battery does not necessarily limit the recorded trip duration, as the bike can remain parked while the rental is still active and you can manually ride the bike if the battery runs out. 

If the dataset included distance travelled we could better investigate whether the bicycle was actively used.

Possible explainations for the long duration
- The user forgot to return the bike or didn't properly complete the return process.
- The bike was parked for several hours while the rental was still active. 
- The recorded timestamp contains an error. 
- Maybe others?

In [ ]:
print("Over 3 hours:", trips_silver.filter(col("duration") > 3 * 60 * 60).count())
print("Over 6 hours:", trips_silver.filter(col("duration") > 6 * 60 * 60).count())
print("Over 12 hours:", trips_silver.filter(col("duration") > 12 * 60 * 60).count())

In [ ]:
total_trips = trips_silver.count()

long_trips = trips_silver.filter(
    col("duration") > 3 * 3600 
).count()

print(f"{long_trips / total_trips * 100}%")

(3184/8470330) * 100 = 0.0376 %

Only 0.038% of all bike trips last longer than 3 hours, inducating such trips are extremely rare.

In [ ]:
trips_silver.groupBy(length("started_at").alias("len")).count().orderBy("len").show()
trips_silver.select("started_at", "ended_at", "duration").distinct().limit(10).show(truncate=False)

## 7. Trip Duration analysis

### Number of trips divided by the different cities

In [ ]:
trips_silver.groupBy("city").count().show()

Oslo accounts for approximately 68% of all trips in the dataset, so the analysis of the trips may be dominated by the travel patterns in Oslo.

#### Number of trips devided by city and stations

In [ ]:
trips_silver.groupBy("city", "start_station_id", "end_station_id").count().show(200)

From the table above it looks like the stationIDs might use the same identifications accross the cities. So the ID alone won't be possible to identify which city the trip was in.  

Let's investigate this a bit ferther:

In [ ]:
trips_silver.groupBy("start_station_id").agg(
    countDistinct("city").alias("num_cities")
).filter(
    col("num_cities") > 1
).show()

The investigantion shows that no start station id is shared across cities in this current dataset. I do think including cities when grouping by station IDs to ensure robustness against potential ovelaps in future data and it gives a better overview. 

#### Number of trips divided by city and month

In [ ]:
trips_silver.groupBy("city", "year_month").count().orderBy(
    "city", "year_month"
).show(200)

## 8. Summary and proposed Silver tranformations
Based on this explorations analysis, this is the following transformations we propose for the Silver layer:
- Extract city and year-month information from the source file path.
- Convert trip duration and geographical coordinates to double. 
- Converting start and and timestamps to UTC timestamps.
- Derive Norwegian local timestamps for temporal analysis. 
- Remove duplicate trips records. 
- Perserve station IDs as strings. 
- Perserve missing station descriptions, as they do not affect the main analytical objective.
- Flag trips with a recorded duration of one second. (Maybe even more?)
- Validate recorded durations against the start and end timestamps. 
- Validate geographical coordinates.

## 9. Test silver_trips.py

In [ ]:
import sys
import os

print(os.getcwd)
sys.path.append("../Silver")

from silver_trips import (
    read_raw_trips,
    build_silver_trips,
    trips_quality_metrics
)

In [ ]:
spark.conf.set("spark.sql.session.timeZone", "UTC")

trips_silver_test = build_silver_trips(spark, RAW)

trips_silver_test.printSchema()
trips_silver_test.show(5, truncate=False)

In [ ]:
trips_quality_metrics(spark, RAW)